**Reference:** https://medium.com/@kasimoluwasegun/langchain-in-production-beyond-the-tutorials-e7b7f2506572

1. Keep prompts in separate files - version-controlled in git

2. Memory Management: Token Limits Are Real
- Keep last k messages
- Summarizes old messages - Clever but expensive
- PostgreSQL + Redis Hybrid
  - Redis for active ssession state (fast, temporary)
  - PostgreSQL for conversation history (durable, queryable)
  - Custom memory class that loads last N messgaes from DB

```python
from langchain.memory import BaseMemory
from typing import Any, Dict, List
import redis
import psycopg2

class ProductionMemory(BaseMemory):
    def __init__(self, session_id: str, postgres_conn, redis_conn):
        self.session_id = session_id
        self.postgres = postgres_conn
        self.redis = redis_conn
        self.window_size = 5
    
    def load_memory_variables(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        # Try Redis first (fast)
        cached = self.redis.get(f"session:{self.session_id}")
        if cached:
            return {"history": cached}
        
        # Fall back to Postgres
        cursor = self.postgres.cursor()
        cursor.execute("""
            SELECT message FROM conversation_history 
            WHERE session_id = %s 
            ORDER BY created_at DESC 
            LIMIT %s
        """, (self.session_id, self.window_size))
        
        messages = [row[0] for row in cursor.fetchall()]
        history = "\n".join(reversed(messages))
        
        # Cache in Redis for 1 hour
        self.redis.setex(f"session:{self.session_id}", 3600, history)
        return {"history": history}
    
    def save_context(self, inputs: Dict[str, Any], outputs: Dict[str, str]) -> None:
        message = f"Human: {inputs['input']}\nAI: {outputs['output']}"
        
        # Save to Postgres
        cursor = self.postgres.cursor()
        cursor.execute("""
            INSERT INTO conversation_history (session_id, message) 
            VALUES (%s, %s)
        """, (self.session_id, message))
        self.postgres.commit()
        
        # Invalidate Redis cache
        self.redis.delete(f"session:{self.session_id}")
```

## **3. Error Handling and Retries**
Failures: Network timeouts, Rate limits, Model Overload, Invalid responses
- If Model_1 is down, retry with exponential backoff

4. Cost Controls
- Add max_tokens limit
- Add caching
- Use models as per the task - for eg: dont use gpt4 for simple classification task
- Add Cost Circuit Breaker - Track your spending - Don't deploy LLM systems without monitoring API costs

```python
from functools import lru_cache
import hashlib

# 1. Set reasonable token limits
llm = ChatOpenAI(
    model="gpt-4",
    max_tokens=500,  # Prevent runaway costs
    temperature=0.7
)
# 2. Cache responses for identical inputs
@lru_cache(maxsize=100)
def get_cached_response(prompt_hash: str, prompt: str):
    return chain.run(prompt)
def run_with_cache(prompt: str):
    prompt_hash = hashlib.md5(prompt.encode()).hexdigest()
    return get_cached_response(prompt_hash, prompt)
# 3. Use cheaper models for simple tasks
simple_llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=150)
complex_llm = ChatOpenAI(model="gpt-4", max_tokens=500)
def route_to_model(task_complexity: str, prompt: str):
    if task_complexity == "simple":
        return simple_llm.predict(prompt)
    return complex_llm.predict(prompt)
# 4. Circuit breaker for cost protection
class CostCircuitBreaker:
    def __init__(self, daily_limit_usd: float):
        self.daily_limit = daily_limit_usd
        self.current_spend = 0.0
        self.reset_date = datetime.now().date()
    
    def check_and_record(self, estimated_cost: float):
        if datetime.now().date() > self.reset_date:
            self.current_spend = 0.0
            self.reset_date = datetime.now().date()
        
        if self.current_spend + estimated_cost > self.daily_limit:
            raise Exception(f"Daily cost limit reached: ${self.daily_limit}")
        
        self.current_spend += estimated_cost
breaker = CostCircuitBreaker(daily_limit_usd=20.0)
```

## **5. Write Tests**
- Unit Tests for individual chains
```python
import pytest
from langchain.evaluation import load_evaluator

def test_log_analysis_chain():
    sample_log = "ERROR: Database connection timeout after 30s"
    result = log_chain.run(sample_log)
    
    # Use LangChain's evaluation framework
    evaluator = load_evaluator("criteria", criteria="relevance")
    eval_result = evaluator.evaluate_strings(
        prediction=result,
        input=sample_log
    )
    
    assert eval_result["score"] > 0.7, "Response not relevant enough"
    assert "timeout" in result.lower(), "Didn't identify timeout issue"
```
- Integration tests for full workflows:
```python
def test_devops_assistant_workflow():
    session_id = "test-123"
    
    # Step 1: Analyze logs
    analysis = analyze_logs(session_id, sample_logs)
    assert "root_cause" in analysis
    
    # Step 2: Get suggestions
    suggestions = get_suggestions(session_id, analysis)
    assert len(suggestions) > 0
    
    # Step 3: Verify memory persistence
    history = get_conversation_history(session_id)
    assert len(history) == 2
```
- Regression Test when prompt changes - Keep a "golden dataset" of 50-100 test cases with expected outputs - you can use cosine similarity on embeddings to detect semantic drift
```python
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

client = OpenAI()
def embedding_similarity(text1: str, text2: str) -> float:
    emb1 = client.embeddings.create(input=text1, model="text-embedding-ada-002").data[0].embedding
    emb2 = client.embeddings.create(input=text2, model="text-embedding-ada-002").data[0].embedding
    return cosine_similarity([emb1], [emb2])[0][0]
def test_prompt_regression():
    for test_case in golden_dataset:
        old_output = test_case["expected_output"]
        new_output = chain.run(test_case["input"])
        
        similarity = embedding_similarity(old_output, new_output)
        assert similarity > 0.85, f"Output diverged too much: {similarity}"
```

## **6. Performance Optimization**
- Streaming Responses
- Parallel Execution
- Cache Embeddings
```python
from functools import lru_cache

@lru_cache(maxsize=1000)
def get_embedding(text: str):
    return embeddings_model.embed_query(text)

```
- Use Faster models Strategically - Simple Classification (gpt-3.5), Complex Reasoning (gpt-4.1), Embeddings (text-embedding-ada-002)
- Async All the Things

## **7. Security and Compliance**
- Prevent Prompt Injection
```python
def sanitize_input(user_input: str) -> str:
    # Remove common injection patterns
    dangerous_patterns = [
        r"ignore previous instructions",
        r"system:",
        r"<\|im_start\|>",
        # Add more based on your threat model
    ]
    
    sanitized = user_input
    for pattern in dangerous_patterns:
        sanitized = re.sub(pattern, "", sanitized, flags=re.IGNORECASE)
    
    # Limit length
    return sanitized[:2000]
```
- Output Filtering
```python
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, validator

class SafeResponse(BaseModel):
    analysis: str
    suggestions: List[str]
    
    @validator('analysis', 'suggestions', each_item=True)
    def no_pii(cls, v):
        # Check for patterns that look like PII
        if re.search(r'\b\d{3}-\d{2}-\d{4}\b', v):  # SSN pattern
            raise ValueError("Response contains PII")
        return v
parser = PydanticOutputParser(pydantic_object=SafeResponse)
```
- API Key Management
```python
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not set")
llm = ChatOpenAI(openai_api_key=openai_api_key)
```

## **8. Deployment Considerations**

- Docker Setup
```python
FROM python:3.11-slim

WORKDIR /app
# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
# Copy application
COPY . .
# Health check endpoint
HEALTHCHECK --interval=30s --timeout=3s \
  CMD curl -f http://localhost:8000/health || exit 1
# Run with gunicorn for production
CMD ["gunicorn", "app:app", "-w", "4", "-k", "uvicorn.workers.UvicornWorker", "--bind", "0.0.0.0:8000"]
```
- Environment Variables
```python
# config.py
from pydantic import BaseSettings

class Settings(BaseSettings):
    openai_api_key: str
    postgres_url: str
    redis_url: str
    max_tokens: int = 500
    rate_limit_per_minute: int = 60
    
    class Config:
        env_file = ".env"
settings = Settings()
```
- Health Checks
```python
from fastapi import FastAPI

app = FastAPI()
@app.get("/health")
async def health_check():
    # Check dependencies
    postgres_ok = await check_postgres()
    redis_ok = await check_redis()
    openai_ok = await check_openai_api()
    
    if all([postgres_ok, redis_ok, openai_ok]):
        return {"status": "healthy"}
    
    return {"status": "degraded", "details": {
        "postgres": postgres_ok,
        "redis": redis_ok,
        "openai": openai_ok
    }}, 503
```
- Rate Limiting
```python
from fastapi_limiter import FastAPILimiter
from fastapi_limiter.depends import RateLimiter

@app.post("/chat")
@app.limiter.limit("10/minute")
async def chat_endpoint(message: str):
    return await process_message(message)

```


## **Common Pitfalls**
Let me save you some pain:

Mistake #1: No retry logic on the first deployment.
First production issue cost us 3 hours of debugging at 2 AM. The fix took 15 minutes. Learn from my stupidity: add retries from day one.

Mistake #2: Logging full prompts with user data.
Security review was… uncomfortable. I was logging entire conversations, including email addresses and internal system details. Now I redact PII before logging anything.

Mistake #3: Using GPT-4 for everything.
My first month’s bill: `$487`. After model routing (GPT-3.5 for simple tasks, GPT-4 for complex ones), my bill dropped to $112. Same functionality.

Mistake #4: No conversation length limits.
Users could keep chatting until token limits exploded. One conversation hit 15,000 tokens. Now I have hard limits and summarization after 10 exchanges.

Mistake #5: Assuming LLM output is always valid JSON.
I had code like json.loads(llm_response) with no error handling. Models hallucinate. They return malformed JSON. They add markdown formatting. Always validate and parse defensively.



## **Your Production Readiness Checklist**
Before you deploy, verify:

✅ Error handling with retries (use tenacity)  
✅ Cost monitoring and daily limits  
✅ Logging (with PII redaction)  
✅ Rate limiting (protect your API and your wallet)  
✅ Testing strategy (unit, integration, regression)  
✅ Database for conversation state (not just in-memory)  
✅ Security review (input sanitization, output filtering)  
✅ Performance benchmarks (measure latency, optimize)  
✅ Deployment pipeline (Docker, CI/CD, health checks)  
✅ Rollback plan (version your prompts, keep old models warm)  